# Data Analysis & Research Notebook

**Hypothesis testing · Regression analysis · Algorithmic performance benchmarks**

- Runnable top-to-bottom in Google Colab or local Jupyter: open it, hit *Run all*.
- Reproducible by design: fixed random seed, package versions printed, install handled in one cell.
- To use your own data: save a CSV into `data/` and point the "Load your own data" cell at it.


In [ ]:
# 1) One-command setup (safe to re-run; no-op when already installed)
%pip install -q pandas numpy scipy scikit-learn


In [ ]:
# 2) Imports + reproducibility setup
import numpy as np
import pandas as pd
import scipy.stats as stats
import sklearn
from sklearn import datasets, linear_model, model_selection, metrics, preprocessing
import time, sys

np.random.seed(42)          # fixed seed -> same results every run
print("Python", sys.version.split()[0])
print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", stats.__version__)
print("scikit-learn", sklearn.__version__)


In [ ]:
# 3) Load data
# Built-in California housing (self-contained, no files needed).
# Swap in YOUR data by replacing this cell with the "Load your own data" cell at the bottom.
df = datasets.fetch_california_housing(as_frame=True).frame
print("Shape:", df.shape)
df.head()


In [ ]:
# 4) Data cleaning, normalization, and train/test split
print("Missing values per column:\n", df.isna().sum().sum(), "total missing")

y = df["MedHouseVal"]                 # target
X = df.drop(columns=["MedHouseVal"])  # features

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = preprocessing.StandardScaler()          # normalize features
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# 5) Hypothesis test
# H0: median income does NOT affect median house value.
# Two-sample t-test comparing high-income vs low-income areas.
med = df["MedInc"].median()
high = df.loc[df["MedInc"] >= med, "MedHouseVal"]
low  = df.loc[df["MedInc"] <  med, "MedHouseVal"]

t_stat, p_value = stats.ttest_ind(high, low)
alpha = 0.05
print(f"t-statistic = {t_stat:.3f}  p-value = {p_value:.3e}")
print("Reject H0 (income affects house value):", p_value < alpha)


In [ ]:
# 6) Regression: predict house value from features
model = linear_model.LinearRegression()
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)

rmse = np.sqrt(metrics.mean_squared_error(y_test, pred))
r2 = metrics.r2_score(y_test, pred)
print(f"Linear regression  RMSE = {rmse:.3f}  R2 = {r2:.3f}")

ridge = linear_model.Ridge(alpha=1.0)            # regularized baseline
ridge.fit(X_train_s, y_train)
print(f"Ridge regression   RMSE = {np.sqrt(metrics.mean_squared_error(y_test, ridge.predict(X_test_s))):.3f}")


In [ ]:
# 7) Algorithmic performance benchmark
# Same operation (.mean()) via numpy vs pandas on 10M values; wall-clock comparison.
n = 10_000_000
arr = np.random.rand(n)
s = pd.Series(arr)

t0 = time.perf_counter(); np_mean = arr.mean(); t_np = time.perf_counter() - t0
t0 = time.perf_counter(); pd_mean = s.mean();  t_pd = time.perf_counter() - t0

assert abs(np_mean - pd_mean) < 1e-12          # same result, different runtime
results = pd.DataFrame({"method": ["numpy", "pandas"],
                        "time_seconds": [t_np, t_pd]})
print(results.round(5).to_string(index=False))
print(f"numpy was {t_np / t_pd:.2f}x faster than pandas on .mean()")


## Adapt this to your own data

1. Save your dataset as `data/my_data.csv` inside this repo.
2. Replace cell 3 with:

```python
df = pd.read_csv("data/my_data.csv")
print("Shape:", df.shape)
df.head()
```

3. Adjust the target/feature split in cell 4 (`y = df["your_target_column"]`, `X = df.drop(columns=["your_target_column"])`).

## Reproducibility checklist (what makes this claim credible)

- [x] Fixed random seed (`np.random.seed(42)`)
- [x] Package versions printed
- [x] Install in one cell (`%pip install`)
- [x] No absolute paths; data lives next to the notebook
- [x] Same results every run

## Interview talking points

- **Why a t-test?** Comparing two independent groups on a continuous target; p < 0.05 rejects the null.
- **What does R2 = 0.6 mean?** The model explains ~60% of the variance in house value.
- **Why benchmark?** Choosing numpy over pandas for the hot loop is a 1-2 line change with a measurable effect.


In [ ]:
# 8) (Optional) Load your own data — copy/paste into cell 3 when ready
# df = pd.read_csv("data/my_data.csv")
# print("Shape:", df.shape)
# df.head()
